# 02 - Entrenar Modelo 1 (binario: sana vs enferma) — v3.3.1

Transfer learning con **EfficientNetB1 + input 240×240** (antes B0+224 — saturado en 0.96 acc). B1 es ~50% más grande pero con más capacidad para distinguir hojas en early-stage disease que B0 no captura.

Mejoras v3.3.1:
- **Backbone B1** (era B0) — capacidad ~5.3M → ~7.8M params, native input 240
- **Input 240×240** (era 224) — más detalle de lesiones pequeñas
- **Augmentación balanceada** sin distorsión elástica/óptica (eran demasiado agresivas)
- **Albumentations API moderna** (`std_range`, `num_holes_range`, `Affine`)
- **Fine-tune amplio** (50 layers) + Cosine LR + EMA tracking
- **safe_read_image** ante `OSError: Transport endpoint`

**⚠ Después de re-entrenar**: en `App/lib/main.dart`, al cargar el healthModel pasar `inputSize: 240` (no default 224):

```dart
final healthModel = await TfliteClassifier.load(
  modelAsset: 'assets/models/hs/model.tflite',
  labelsAsset: 'assets/models/hs/labels.txt',
  inputSize: 240,
);
```

Targets: F1 ≥ 0.98, accuracy ≥ 0.98, recall ≥ 0.97 (vs 0.96 actual).

In [ ]:
!pip install -q tensorflow albumentations scikit-learn matplotlib

In [ ]:
import tensorflow as tf
import numpy as np
import json
from pathlib import Path
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)
print("GPU:", tf.config.list_physical_devices("GPU"))

IMG_SIZE = (240, 240)
BATCH = 12
EPOCHS_P1 = 18
EPOCHS_P2 = 40
LR_P1 = 1e-3
LR_P2 = 3e-6
EMA_DECAY = 0.9999

DATA = Path("./splits/clasificacion_binaria")
OUT = Path("./outputs")
OUT.mkdir(exist_ok=True)

In [ ]:
import time
import numpy as np
import albumentations as A
from PIL import Image
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications.efficientnet import preprocess_input


TRAIN_AUG = A.Compose([
    A.Rotate(limit=40, border_mode=0, p=0.6),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
    A.HueSaturationValue(hue_shift_limit=12, sat_shift_limit=25, val_shift_limit=15, p=0.4),
    A.CLAHE(clip_limit=2.5, tile_grid_size=(8, 8), p=0.3),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        A.MotionBlur(blur_limit=5, p=1.0),
    ], p=0.2),
    A.GaussNoise(std_range=(0.02, 0.08), p=0.2),
    A.Affine(translate_percent=0.06, scale=(0.9, 1.12), rotate=0, p=0.3),
    A.CoarseDropout(
        num_holes_range=(1, 4),
        hole_height_range=(12, 24),
        hole_width_range=(12, 24),
        fill=0,
        p=0.15,
    ),
])

VAL_AUG = A.Compose([])


def safe_read_image(fp, target_size, retries=3, delay=0.5):
    for attempt in range(retries):
        try:
            return np.array(Image.open(fp).convert("RGB").resize(target_size))
        except (OSError, IOError) as exc:
            if attempt < retries - 1:
                time.sleep(delay)
                continue
            print(f"[warn] saltando {fp}: {exc}")
            return None


class LeafSequence(Sequence):
    def __init__(self, directory, img_size=(224, 224), batch_size=16,
                 augment=False, class_mode="binary", shuffle=True):
        self.img_size = img_size
        self.batch_size = batch_size
        self.augment = augment
        self.class_mode = class_mode
        self.shuffle = shuffle
        self.samples = []
        self.class_indices = {}
        classes = sorted(p.name for p in Path(directory).iterdir() if p.is_dir())
        for i, cls in enumerate(classes):
            self.class_indices[cls] = i
            for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
                for fp in (Path(directory) / cls).glob(ext):
                    self.samples.append((str(fp), i))
        self.n = len(self.samples)
        self.classes = np.array([s[1] for s in self.samples])
        if shuffle:
            self._shuffle()

    def _shuffle(self):
        idx = np.random.permutation(len(self.samples))
        self.samples = [self.samples[i] for i in idx]
        self.classes = self.classes[idx]

    def __len__(self):
        return max(1, self.n // self.batch_size)

    def __getitem__(self, i):
        batch = self.samples[i * self.batch_size:(i + 1) * self.batch_size]
        imgs, labels = [], []
        for fp, label in batch:
            img = safe_read_image(fp, self.img_size)
            if img is None:
                continue
            if self.augment:
                img = TRAIN_AUG(image=img)["image"]
            else:
                img = VAL_AUG(image=img)["image"]
            img = preprocess_input(img.astype(np.float32))
            imgs.append(img)
            labels.append(label)
        if not imgs:
            placeholder = preprocess_input(np.zeros((1, *self.img_size, 3), dtype=np.float32))
            placeholder_y = np.zeros((1,), dtype=np.float32) if self.class_mode == "binary" else np.zeros((1, len(self.class_indices)), dtype=np.float32)
            return placeholder, placeholder_y
        X = np.stack(imgs)
        if self.class_mode == "binary":
            Y = np.array(labels, dtype=np.float32)
        else:
            n_cls = len(self.class_indices)
            Y = np.eye(n_cls)[labels]
        return X, Y

    def on_epoch_end(self):
        if self.shuffle:
            self._shuffle()


train_gen = LeafSequence(
    DATA / "train", img_size=IMG_SIZE, batch_size=BATCH,
    augment=True, class_mode="binary", shuffle=True,
)
val_gen = LeafSequence(
    DATA / "val", img_size=IMG_SIZE, batch_size=BATCH,
    augment=False, class_mode="binary", shuffle=False,
)
print(f"Train: {train_gen.n} imagenes | Val: {val_gen.n} imagenes")
print("Clases:", train_gen.class_indices)
with open(OUT / "class_indices_model1_binary.json", "w") as f:
    json.dump(train_gen.class_indices, f)

In [ ]:
def construir(num_clases=1):
    base = tf.keras.applications.EfficientNetB1(
        weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3)
    )
    base.trainable = False
    x = base.output
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(
        256, activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4)
    )(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    if num_clases == 1:
        out = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    else:
        out = tf.keras.layers.Dense(num_clases, activation="softmax")(x)
    return tf.keras.Model(base.input, out, name="model1_binary_b1")


model = construir()
loss = "binary_crossentropy"
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_P1),
    loss=loss,
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="prec"),
        tf.keras.metrics.Recall(name="rec"),
    ],
)
model.summary()

In [ ]:
cbs_p1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=6, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(OUT / "model1_binary_p1_best.keras"),
        monitor="val_accuracy", save_best_only=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1
    ),
]

print("=== FASE 1: cabeza (base congelada) ===")
h1 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS_P1, callbacks=cbs_p1, verbose=1,
)


In [ ]:
class EMACallback(tf.keras.callbacks.Callback):
    def __init__(self, decay=0.9999):
        super().__init__()
        self.decay = decay
        self.ema_weights = None

    def on_train_begin(self, logs=None):
        self.ema_weights = [w.numpy().copy() for w in self.model.trainable_weights]

    def on_train_batch_end(self, batch, logs=None):
        for i, w in enumerate(self.model.trainable_weights):
            self.ema_weights[i] = self.decay * self.ema_weights[i] + (1 - self.decay) * w.numpy()

    def on_train_end(self, logs=None):
        for w, ew in zip(self.model.trainable_weights, self.ema_weights):
            w.assign(ew)


for layer in model.layers:
    layer.trainable = True
for layer in model.layers[:-50]:
    layer.trainable = False
for layer in model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainables = sum(1 for l in model.layers if l.trainable)
print(f"Layers entrenables: {trainables}")

steps_per_epoch = max(1, train_gen.n // BATCH)
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=LR_P2,
    decay_steps=EPOCHS_P2 * steps_per_epoch,
    alpha=0.05,
)
model.compile(
    optimizer=tf.keras.optimizers.Adam(lr_schedule),
    loss=loss,
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="prec"),
        tf.keras.metrics.Recall(name="rec"),
    ],
)

cbs_p2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=12, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(OUT / "model1_binary_p2_best.keras"),
        monitor="val_accuracy", save_best_only=True, verbose=1,
    ),
    EMACallback(decay=EMA_DECAY),
]

print("=== FASE 2: fine-tuning ultimos 50 layers + Cosine LR + EMA ===")
h2 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS_P2, initial_epoch=len(h1.history["loss"]),
    callbacks=cbs_p2, verbose=1,
)

model.save(OUT / "model1_binary.keras")
print("Modelo guardado en", OUT / "model1_binary.keras")

In [ ]:
acc = h1.history["accuracy"] + h2.history["accuracy"]
vacc = h1.history["val_accuracy"] + h2.history["val_accuracy"]
loss_h = h1.history["loss"] + h2.history["loss"]
vloss = h1.history["val_loss"] + h2.history["val_loss"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
div = len(h1.history["accuracy"])
ep = range(1, len(acc) + 1)
for ax, t, v, ylabel in [(ax1, acc, vacc, "Accuracy"), (ax2, loss_h, vloss, "Loss")]:
    ax.plot(ep, t, "b-", label="train")
    ax.plot(ep, v, "r-", label="val")
    ax.axvline(div, color="gray", linestyle="--", label="inicio fine-tune")
    ax.set_xlabel("Epoca"); ax.set_ylabel(ylabel); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "model1_binary_curves.png", dpi=120)
plt.show()
